# OpenShell UI on Google Colab

Runs the dashboard inside the notebook runtime and renders it in an iframe.

**No public tunnel is started on purpose.** The Colab FAQ prohibits file hosting or
web service offerings unrelated to interactive compute, and for free tiers prohibits
bypassing the notebook UI to interact primarily through a web UI, or connecting to
remote proxies. A tunnel also breaks whenever the VM is reset, and the runtime is
terminated after an idle period.

To reach a public URL, deploy the same app to a Cloudflare Tunnel or Vercel and use
this notebook only as the sandbox-side runtime.

In [ ]:
import subprocess
import sys

REPO = "https://github.com/watanabe3tipapa/openshell-plus"
PORT = 8080

# このリポジトリが public かつ push 済みでないとインストールできない。
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", f"git+{REPO}"],
    check=False,
)
if result.returncode != 0:
    raise SystemExit(
        f"pip install failed for {REPO}. "
        "The repository must be public and pushed before this cell can work."
    )
print("installed openshell-ui from GitHub")


## Run mode

Colab has no Docker daemon and no OpenShell gateway, so the app starts in demo mode.
Set `OSUI_DEMO=true` explicitly to make that visible instead of relying on the probe.

In [ ]:
import os

os.environ["OSUI_DEMO"] = "true"
os.environ["OSUI_MODE"] = "colab"
os.environ["OSUI_HOST"] = "127.0.0.1"
os.environ["OSUI_PORT"] = str(PORT)

server = subprocess.Popen(
    ["uvicorn", "openshell_ui.main:app", "--host", "127.0.0.1", "--port", str(PORT)],
    env={**os.environ},
)
print(f"uvicorn pid={server.pid} on 127.0.0.1:{PORT}")

In [ ]:
import socket
import time

for _ in range(60):
    with socket.socket() as sock:
        if sock.connect_ex(("127.0.0.1", PORT)) == 0:
            print("dashboard is accepting connections")
            break
    time.sleep(0.5)
else:
    raise RuntimeError("dashboard did not start")

## Render in the notebook

`serve_kernel_port_as_iframe` is an internal `google.colab.output._util` helper, not a
supported public API. Colab publishes no public port-forwarding API and no official
documentation endorses exposing a port.

The cell below embeds the dashboard in an iframe so the notebook UI stays in control.
If the internal iframe helper disappears, it falls back to the officially exported
`serve_kernel_port_as_window`, which opens a separate browser tab instead.

In [ ]:
# Prefer the internal iframe helper so the notebook UI stays the primary surface.
# Fall back to the officially exported window helper (opens a new tab) if it is gone.
try:
    from google.colab.output._util import serve_kernel_port_as_iframe
except ImportError:
    from google.colab.output import serve_kernel_port_as_window

    serve_kernel_port_as_window(PORT, path="/")
else:
    serve_kernel_port_as_iframe(PORT, path="/", width="100%", height="760")


## Not supported: public tunnel

The next cell is intentionally inert. Uncommenting it exposes an unauthenticated
runtime to the internet in violation of the Colab usage policy, and the URL dies with
the VM. Use `scripts/run-cloudflare.sh quick` on a local machine instead.

In [ ]:
# subprocess.run(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"], check=False)